# Day 24 — Attention visualization

This notebook extracts the final encoder/decoder layer's per-head attention from the trained PyTorch and TensorFlow checkpoints. The plotted maps average four heads for readability; `results/day24/attention.json` retains every head. The decoder self-attention panel should be strictly causal, while decoder cross-attention shows which English source tokens support each generated German token.

In [ ]:
from pathlib import Path
import json
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(path for path in candidates if (path / 'roadmap.md').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

## Extract, validate, and render

PyTorch uses CUDA. TensorFlow uses CPU because current native Windows TensorFlow releases do not provide CUDA support; WSL2/Linux can use `--tf-device gpu`.

In [ ]:
from visualization.attention import main

main([
    '--examples', '3',
    '--torch-device', 'cuda',
    '--tf-device', 'cpu',
    '--output-dir', str(ROOT / 'results' / 'day24'),
])

## Inspect translations and matrix dimensions

In [ ]:
records = json.loads((ROOT / 'results/day24/attention.json').read_text(encoding='utf-8'))
[{
    'framework': row['framework'],
    'source': row['source'],
    'translation': row['translation'],
    'heads': len(row['decoder_cross']),
    'cross_attention_shape': (
        len(row['decoder_cross']),
        len(row['target_tokens']),
        len(row['source_tokens']),
    ),
} for row in records]

## Display the generated heatmaps

In [ ]:
from IPython.display import Image, display

for framework in ('pytorch', 'tensorflow'):
    for example in range(1, 4):
        display(Image(filename=str(
            ROOT / 'results/day24' / f'{framework}-example-{example}.png'
        )))